# 창고 로봇 YOLO11n 학습 (Colab)

M2 예제 4의 전체 학습을 Colab GPU(T4)에서 수행한다. 준비물: `datasets/warehouse` 폴더를 zip 으로 묶은 `warehouse.zip` (03번 예제 출력. `cd datasets && zip -r warehouse.zip warehouse`). 런타임 유형을 **GPU** 로 바꾼 뒤 위에서부터 실행한다. 약 3천 장 x 60 epoch 은 T4 에서 30~40분.

In [ ]:
!pip -q install ultralytics
import torch, ultralytics
print(torch.__version__, torch.cuda.is_available(), ultralytics.__version__)

## 1. 데이터셋 올리기
파일을 직접 업로드하거나(느림) Google Drive 에 올려 두고 마운트한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Drive 의 경로를 자신의 것으로 바꾼다
!cp /content/drive/MyDrive/warehouse.zip /content/ && cd /content && unzip -q -o warehouse.zip
!sed -i 's#^path:.*#path: /content/warehouse#' /content/warehouse/data.yaml && cat /content/warehouse/data.yaml

## 2. 학습
04번 스크립트와 같은 하이퍼파라미터다. 시뮬레이터 영상은 색·조명이 단조로워 색 증강을 조금 세게 준다.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data='/content/warehouse/data.yaml', epochs=60, imgsz=640, batch=32, device=0, workers=2,
            project='runs/m2', name='colab_full_640', exist_ok=True,
            hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, scale=0.5, fliplr=0.5, mosaic=1.0, close_mosaic=10)

## 3. 평가와 내보내기
CPU 배포용으로 ONNX(320 입력)도 함께 만든다. 06번 예제에서 OpenVINO·INT8 로 더 최적화한다.

In [ ]:
best = 'runs/m2/colab_full_640/weights/best.pt'
m = YOLO(best)
r = m.val(data='/content/warehouse/data.yaml', imgsz=640, device=0)
print('mAP50', r.box.map50, 'mAP50-95', r.box.map)
m.export(format='onnx', imgsz=320, opset=17, simplify=True, dynamic=False)

In [ ]:
from google.colab import files
import shutil
shutil.copy(best, 'warehouse_yolo11n.pt')
files.download('warehouse_yolo11n.pt')
files.download('runs/m2/colab_full_640/weights/best.onnx')

받은 `warehouse_yolo11n.pt` 를 저장소의 `weights/` 에 넣으면 05~07 예제가 그대로 돈다.